In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# All-NBA Rosters All time

In [ ]:
def scrape_all_nba_teams():
    """
    Scrape All-NBA team data from NBA.com
    Returns a pandas DataFrame with season, player_name, and team columns
    """
    url = "https://www.nba.com/news/history-all-nba-teams"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    # Find all content - the page has season headers followed by team listings
    content = soup.find("div", {"class": "ArticleContent_articleContent__AnSpt"})

    if not content:
        # Try alternative selector
        content = soup.find("article") or soup

    data = []
    current_season = None

    # Find all text content
    text = content.get_text()

    # Split by season markers (look for year patterns like "2024-25" or "> 2024-25")
    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        # Check if this is a season header (e.g., "> 2024-25" or "2024-25")
        season_match = re.match(r">?\s*(\d{4}-\d{2,4})", line)
        if season_match:
            current_season = season_match.group(1)
            continue

        # Skip empty lines, team headers, and notes
        if not line or line in [
            "FIRST TEAM",
            "SECOND TEAM",
            "THIRD TEAM",
            "F:",
            "G:",
            "C:",
            "•",
        ]:
            continue

        if (
            line.startswith("*")
            or line.startswith("Note:")
            or line.startswith("Official")
        ):
            continue

        # Parse player lines (format: "• Name, Team" or "• F: Name, Team" or "• Position: Name, Team")
        if current_season and (
            "," in line or "F:" in line or "G:" in line or "C:" in line
        ):
            # Remove bullet points and position markers
            cleaned = re.sub(r"^[•\s]*[FGC]:\s*", "", line)
            cleaned = cleaned.replace("•", "").strip()

            # Split by comma to separate player from team
            if "," in cleaned:
                parts = cleaned.rsplit(",", 1)
                if len(parts) == 2:
                    player_name = parts[0].strip()
                    team_name = parts[1].strip()

                    # Skip if it's not a valid player entry
                    if player_name and team_name and len(player_name) > 2:
                        data.append(
                            {
                                "season": current_season,
                                "player_name": player_name,
                                "team": team_name,
                            }
                        )

    df = pd.DataFrame(data)

    return df

In [ ]:
if __name__ == "__main__":
    try:
        print("Scraping All-NBA teams from NBA.com...")
        df = scrape_all_nba_teams()

        print(f"\nSuccessfully scraped {len(df)} All-NBA selections")
        print(f"\nSeasons covered: {df['season'].nunique()}")
        print(f"\nFirst few rows:")
        print(df.head(20))

        # Save to CSV
        output_file = "all_nba_teams.csv"
        df.to_csv(output_file, index=False)
        print(f"\nData saved to {output_file}")

        # Show sample statistics
        print(f"\nPlayers by season (last 5 seasons):")
        print(df.groupby("season").size().tail())

    except Exception as e:
        print(f"Error: {e}")
        import traceback

        traceback.print_exc()

Scraping All-NBA teams from NBA.com...

Successfully scraped 976 All-NBA selections

Seasons covered: 79

First few rows:
     season              player_name                    team
0   2024-25    Giannis Antetokounmpo         Milwaukee Bucks
1   2024-25  Shai Gilgeous-Alexander   Oklahoma City Thunder
2   2024-25             Nikola Jokić          Denver Nuggets
3   2024-25         Donovan Mitchell     Cleveland Cavaliers
4   2024-25             Jayson Tatum          Boston Celtics
5   2024-25            Jalen Brunson         New York Knicks
6   2024-25            Stephen Curry   Golden State Warriors
7   2024-25          Anthony Edwards  Minnesota Timberwolves
8   2024-25             LeBron James      Los Angeles Lakers
9   2024-25              Evan Mobley     Cleveland Cavaliers
10  2024-25          Cade Cunningham         Detroit Pistons
11  2024-25        Tyrese Haliburton          Indiana Pacers
12  2024-25             James Harden             LA Clippers
13  2024-25       Karl-A

# All-Star Rosters

### Adjust the start and end years to get wanted range

In [ ]:
def scrape_allstar_rosters(start_year=2015, end_year=2025):
    """
    Scrape NBA All-Star rosters from Basketball Reference
    Includes all players selected (whether they played or not)

    Args:
        start_year: First year to scrape (e.g., 2015 for 2014-15 season)
        end_year: Last year to scrape (e.g., 2025 for 2024-25 season)

    Returns:
        DataFrame with season, player_name, and team columns
    """

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    all_data = []

    for year in range(start_year, end_year + 1):
        # Convert year to season format (e.g., 2015 -> "2014-15")
        season = f"{year-1}-{str(year)[2:]}"

        url = f"https://www.basketball-reference.com/allstar/NBA_{year}.html"

        try:
            print(f"Scraping {season} season ({year})...")
            response = requests.get(url, headers=headers)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, "html.parser")

            # Find both East and West roster tables
            tables = soup.find_all("table", {"class": "stats_table"})

            for table in tables:
                # Check if this is a roster table (not box score)
                table_id = table.get("id", "")
                if "box" in table_id.lower():
                    continue

                tbody = table.find("tbody")
                if not tbody:
                    continue

                rows = tbody.find_all("tr")

                for row in rows:
                    # Skip header rows
                    if row.find("th", {"class": "over_header"}):
                        continue

                    # Get player name from the data-append-csv attribute or th tag
                    player_cell = row.find("th", {"data-stat": "player"})
                    if not player_cell:
                        continue

                    player_name = player_cell.get_text(strip=True)

                    # Get team from team_id column
                    team_cell = row.find("td", {"data-stat": "team_id"})
                    if team_cell:
                        team_name = team_cell.get_text(strip=True)
                    else:
                        # Fallback: try to find team in the player link
                        link = player_cell.find("a")
                        if link:
                            team_name = ""  # Will need to extract from somewhere else
                        else:
                            team_name = ""

                    # Skip if we don't have essential data
                    if not player_name:
                        continue

                    all_data.append(
                        {
                            "season": season,
                            "player_name": player_name,
                            "team": team_name,
                        }
                    )

        except Exception as e:
            print(f"Error scraping {season}: {e}")
            continue

    df = pd.DataFrame(all_data)

    # Remove duplicates (same player might appear in multiple tables)
    df = df.drop_duplicates(subset=["season", "player_name"])

    return df

In [ ]:
# Main execution
if __name__ == "__main__":
    try:
        print("Scraping NBA All-Star rosters from 2014-15 season onwards...")
        df = scrape_allstar_rosters(start_year=2015, end_year=2025)

        print(f"\nSuccessfully scraped {len(df)} All-Star selections")
        print(f"\nSeasons covered: {sorted(df['season'].unique())}")
        print(f"\nFirst few rows:")
        print(df.head(20))

        # Save to CSV
        output_file = "nba_allstar_rosters.csv"
        df.to_csv(output_file, index=False)
        print(f"\nData saved to {output_file}")

        # Show statistics
        print(f"\nAll-Stars per season:")
        print(df.groupby("season").size().sort_index())

    except Exception as e:
        print(f"Error: {e}")
        import traceback

        traceback.print_exc()

Scraping NBA All-Star rosters from 2014-15 season onwards...
Scraping 2014-15 season (2015)...
Scraping 2015-16 season (2016)...
Scraping 2016-17 season (2017)...
Scraping 2017-18 season (2018)...
Scraping 2018-19 season (2019)...
Scraping 2019-20 season (2020)...
Scraping 2020-21 season (2021)...
Scraping 2021-22 season (2022)...
Scraping 2022-23 season (2023)...
Scraping 2023-24 season (2024)...
Scraping 2024-25 season (2025)...

Successfully scraped 264 All-Star selections

Seasons covered: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

First few rows:
     season        player_name team
0   2014-15       James Harden  HOU
1   2014-15      Stephen Curry  GSW
2   2014-15         Marc Gasol  MEM
3   2014-15      Klay Thompson  GSW
4   2014-15  LaMarcus Aldridge  POR
5   2014-15           Reserves     
6   2014-15         Chris Paul  LAC
7   2014-15  Russell Westbrook  OKC
8   2014-15   DeMarcus Cousins  SAC
9 